# *__Libraries for working with dash__*

In [ ]:
! pip3 install PyArrow
! pip3 install plotly
! pip3 install plotly_express
! pip3 install raceplotly
! pip3 install kaleido
! pip install dash==2.11.0
! pip3 install Werkzeug <= 2.1.2
! pip3 install -U flask==2.1.3
! pip3 install jupyter-dash
! pip3 install dash-bootstrap-components

In [ ]:
#Libreria para manejo de archivos json
import json
#Librerias para trabajar con visualizaciones interactivas
import plotly
import plotly.express as px
#Librerias para crear un tablero con plotly
from jupyter_dash import JupyterDash
from dash import dcc
from dash import html
import dash
#Libreria para cambiar el diseño de un tablero en dash
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output, State
#Libreria para manipulacion de datos con dataframes
import pandas as pd
import numpy as np
#Para manipulacion de fechas en controles de dash
from datetime import date
#Para manipulacion de archivos cargados desde un cliente
import base64
import datetime
import io

# *__Libraries for working with model training and prediction__*

In [ ]:
! pip install category_encoders

In [ ]:
#Librería para aplicar target encoding
from category_encoders import TargetEncoder
#Librería para estandarizar los datos
from sklearn.preprocessing import StandardScaler
#Librería para dividir el dataset en datos de entrenamiento y de prueba
from sklearn.model_selection import train_test_split
#Librerías para aplicar los modelos y evaluarlos
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
#Librería para guardar los modelos, los codificadores y el escalador
import joblib
#Librería para conectar con Google Drive
from google.colab import drive

# ***Model Training***

## Loading Modeling Dataset

In [ ]:
drive.mount('/content/drive')
path='/content/drive/Shareddrives/COLAB DATOS DE LIVERPOOL/Datos Liverpool/'
Dataset_2=pd.read_csv(path+'DatasetModelado.csv')

In [ ]:
Dataset_2

In [ ]:
#Hacemos drop
Dataset_2 = Dataset_2.drop(columns='Nº pers.')

## Target Encoding (''Antigüedad en Grupos'')

In [ ]:
#target encoding para la columna 'GroPer'
encoder1 = TargetEncoder(cols = ['Gpo Personal'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Gpo_Personal_TE'] = encoder1.fit_transform(Dataset_2['Gpo Personal'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Gpo Personal', axis=1)

In [ ]:
#target encoding para la columna 'Departamento'
encoder2 = TargetEncoder(cols = ['Departamento'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Departamento_TE'] = encoder2.fit_transform(Dataset_2['Departamento'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Departamento', axis=1)

In [ ]:
#target encoding para la columna 'Desc Fun'
encoder3 = TargetEncoder(cols = ['Función'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Función_TE'] = encoder3.fit_transform(Dataset_2['Función'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Función', axis=1)

In [ ]:
#target encoding para la columna 'Locación'
encoder4 = TargetEncoder(cols = ['Ubicación'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Ubicación_TE'] = encoder4.fit_transform(Dataset_2['Ubicación'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Ubicación', axis=1)

In [ ]:
#target encoding para la columna 'Locación'
encoder5 = TargetEncoder(cols = ['Área Personal'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Área_Personal_TE'] = encoder5.fit_transform(Dataset_2['Área Personal'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Área Personal', axis=1)

In [ ]:
#target encoding para la columna 'Locación'
encoder6 = TargetEncoder(cols = ['Desc Soc'], smoothing=0)

# Fit and transform the encoder on your dataset
Dataset_2['Desc_Soc_TE'] = encoder6.fit_transform(Dataset_2['Desc Soc'], Dataset_2['Antigüedad_Grupos'])

# Drop the original column
Dataset_2 = Dataset_2.drop('Desc Soc', axis=1)

In [ ]:
Dataset_2 = Dataset_2.dropna()        # Drop de todas las filas con valores NaN
Dataset_2 = Dataset_2.reset_index(drop=True)
Dataset_2

## Model Training and Saving

### Multinomial Logistic Regression ('Antigüedad en Grupos')

In [ ]:
X = Dataset_2.drop(columns=['Antigüedad_Grupos'], axis=1)
y = Dataset_2['Antigüedad_Grupos']

In [ ]:
#Dividir el dataset en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#Estandarizar los features para poder realizar la regresi+on
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Entrenar el modelo aplicando Regresión Logística Multinomial
clf_LR = LogisticRegression(multi_class='multinomial', solver='lbfgs')
clf_LR.fit(X_train_scaled, y_train)

In [ ]:
#Predecimos en los datos de prueba
y_pred = clf_LR.predict(X_test_scaled)

In [ ]:
#Evaluar el modelo
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

In [ ]:
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

#Imprimimos los resultados
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')

In [ ]:
#Generar un reporte de clasificación
report = classification_report(y_test, y_pred)

print("Classification Report:\n", report)

### Random Forest ('Antigüedad en Grupos')

In [ ]:
X = Dataset_2.drop(columns=['Antigüedad_Grupos'], axis=1)
y = Dataset_2['Antigüedad_Grupos']

In [ ]:
#Dividir el dataset en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#Entrenar el modelo aplicando Random Forest
clf_RF = RandomForestClassifier(n_estimators=70, random_state=42)
clf_RF.fit(X_train, y_train)

In [ ]:
#Predecimos en los datos de prueba
y_pred = clf_RF.predict(X_test)

In [ ]:
#Evaluar el modelo
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

In [ ]:
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

#Imprimimos los resultados
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')

In [ ]:
#Generar un reporte de clasificación
report = classification_report(y_test, y_pred)

print("Classification Report:\n", report)

### Saving Classifiers, Scalers, and Encoders

In [ ]:
#Guardamos el modelo utilizando joblib
joblib.dump(clf_RF, 'RandomForest_clf.joblib')

In [ ]:
#Guardamos el escalador utilizando joblib
joblib.dump(scaler, 'standard_scaler.joblib')

In [ ]:
joblib.dump(clf_LR, 'LogisticRegression_clf.joblib')

In [ ]:
#Guardamos el target encoder utilizando joblib
joblib.dump(encoder1, 'target_encoder_Gpo_Personal.joblib')

In [ ]:
joblib.dump(encoder2, 'target_encoder_Departamento.joblib')

In [ ]:
joblib.dump(encoder3, 'target_encoder_Funcion.joblib')

In [ ]:
joblib.dump(encoder4, 'target_encoder_Ubicacion.joblib')

In [ ]:
joblib.dump(encoder5, 'target_encoder_Area_Personal.joblib')

In [ ]:
joblib.dump(encoder6, 'target_encoder_Desc_Soc.joblib')

# ***Features for Future Liverpool Predictions***

In [ ]:
def transform_and_drop_column(dataset, column_name, encoder_filename, target_column_name):
    #Cargar target encoding
    target_encoder = joblib.load(encoder_filename)

    #Transformar la columna categórica
    dataset[target_column_name] = target_encoder.transform(dataset[column_name])

    #Hacer drop en la columna original
    dataset = dataset.drop(column_name, axis=1)

    return dataset

#Lista de transformaciones
transformations = [
    ('Gpo Personal', 'target_encoder_Gpo_Personal.joblib', 'Gpo_Personal_TE'),
    ('Departamento', 'target_encoder_Departamento.joblib', 'Departamento_TE'),
    ('Función', 'target_encoder_Funcion.joblib', 'Función_TE'),
    ('Ubicación', 'target_encoder_Ubicacion.joblib', 'Ubicación_TE'),
    ('Área Personal', 'target_encoder_Area_Personal.joblib', 'Área_Personal_TE'),
    ('Desc Soc', 'target_encoder_Desc_Soc.joblib', 'Desc_Soc_TE')
]

In [ ]:
#Definir un mapping para cambiar clases numéricas a strings
class_mapping = {0: 'en menos de 1 año', 1: 'entre 1 y 5 años', 2: 'en más de 5 años'}

#Definir nombres para cada clase
class_labels = ['Menos de un año', 'Entre 1 y 5 años', 'Más de 5 años']

#Cargar clasificadores ya entrenados y el escalador
RandomForest_clf = joblib.load('RandomForest_clf.joblib')
standard_scaler = joblib.load('standard_scaler.joblib')
LogisticRegression_clf = joblib.load('LogisticRegression_clf.joblib')

In [ ]:
def predict_and_display_results(DatasetLiverpool, transformations, standard_scaler, LogisticRegression_clf, RandomForest_clf, class_mapping, class_labels):
    #Aplicar target encoding a columnas categóricas y hacer drop a las columnas originales
    for transformation in transformations:
        DatasetLiverpool = transform_and_drop_column(DatasetLiverpool, *transformation)

    #Estandarizar los datos
    scaled_data = standard_scaler.transform(DatasetLiverpool)

    #Predecir la duración del empleado y mostrarla en texto con ambos clasificadores
    predictions_LR_numeric = LogisticRegression_clf.predict(scaled_data)
    predictions_LR_string = [class_mapping[pred] for pred in predictions_LR_numeric]
    predictions_LR_text = ' '.join(predictions_LR_string)

    predictions_RF_numeric = RandomForest_clf.predict(DatasetLiverpool)
    predictions_RF_string = [class_mapping[pred] for pred in predictions_RF_numeric]
    predictions_RF_text = ' '.join(predictions_RF_string)

    text1=html.H5(["Primer Pronóstico:",html.Br(),"El empleado renunciará", html.Br(), predictions_RF_text])
    text3=html.H5(["Segundo Pronóstico:",html.Br(),"El empleado renunciará", html.Br(), predictions_LR_text])

    #Predecir las probabilidades de las clases y mostrarlas en texto con ambos clasificadores
    predicted_probabilities_LR = LogisticRegression_clf.predict_proba(scaled_data)
    class_prob_strings_LR = []
    for probs in predicted_probabilities_LR:
        class_probs = [f"{class_labels[i]}: {prob * 100:.2f}%" for i, prob in enumerate(probs)]
        class_prob_strings_LR.append(class_probs)
    class_prob_texts_LR = ['\n'.join(class_probs) for class_probs in class_prob_strings_LR]

    predicted_probabilities_RF = RandomForest_clf.predict_proba(DatasetLiverpool)
    class_prob_strings_RF = []
    for probs in predicted_probabilities_RF:
        class_probs2 = [f"{class_labels[i]}: {prob * 100:.2f}%" for i, prob in enumerate(probs)]
        class_prob_strings_RF.append(class_probs2)
    class_prob_texts_RF = ['\n'.join(class_probs2) for class_probs2 in class_prob_strings_RF]

    text2=html.H5(["Probabilidades 1:",html.Br(), *class_prob_texts_RF])
    text4=html.H5(["Probabilidades 2:",html.Br(), *class_prob_texts_LR])

    #Encontrar la clase con mayor probabilidad considerando ambos clasificadores
    max_prob_classes = [
        class_labels[max(lr_prob.argmax(), rf_prob.argmax())]
        for lr_prob, rf_prob in zip(predicted_probabilities_LR, predicted_probabilities_RF)
    ]

    text5 = html.H5(["Pronóstico Definitivo",html.Br(),"(Probabilidad Más Alta):", html.Br(), *max_prob_classes])

    return text1, text2, text3, text4, text5

# ***Interactive Dashboard***

## Loading Clean Demo Dataset for Working with Charts

In [ ]:
path='/content/drive/Shareddrives/COLAB DATOS DE LIVERPOOL/Datos Liverpool/'
Demo=pd.read_csv(path+'DemoLimpio.csv')

In [ ]:
#Reducimos el dataframe conservando las columnas relevantes para que cargue más rápido
columns_to_drop = ['NAño','NMes','NDia','GroPer','Área Personal','Unidad','Desc Soc','Ultima Evaluación', 'Edad ingreso', 'CP Vivienda', 'IAño', 'CP Trabajo', 'Nº pers.', 'Fecha nacimiento', 'Fecha ingreso', 'Fecha Salida', 'Año salida', 'Desc Medida', 'Genero', 'Edad salida','IMes', 'Duración Días','Antigüedad_Grupos']
Demo = Demo.drop(columns=columns_to_drop)
Demo

,No Hijos,SAño,SMes,Antigüedad,Locación,Desc Fun,Departamento
0,0,2022,12,7,Suburbia Los Cabos Patio,Cajero,CAJAS
1,0,2021,10,2,Boutique Los Cabos,Vendedor Bilingue,FRAGANCIAS
2,2,2022,12,3,Boutique Los Cabos,Consejero de Belleza,FRAGANCIAS
3,0,2020,2,7,Boutique Los Cabos,Vendedor Bilingue,FRAGANCIAS
4,0,2022,11,4,Boutique Los Cabos,Consejero de Belleza,FRAGANCIAS
...,...,...,...,...,...,...,...
61459,0,2019,4,1,Sfera Tlaquepaque,Vendedor Boutique Sfera,BOUTIQUES
61460,0,2019,1,0,Sfera Playa del Carmen,Vendedor Boutique Sfera,BOUTIQUES
61461,0,2020,1,0,Sfera Guadalajara Gran Plaza,Vendedor Boutique Sfera,BOUTIQUES
61462,0,2020,1,0,Sfera Galerias Lag Torreon,Vendedor Boutique Sfera,BOUTIQUES


In [ ]:
column_name = 'SAño'
value_to_drop = 2023

#Drop de filas donde aparece el año 2023
Demo = Demo[Demo[column_name] != value_to_drop]

## Functions for working with charts

In [ ]:
#Valores únicos de las columnas a incluir dentro del tablero
locacion_options = Demo['Locación'].unique()
funcion_options = Demo['Desc Fun'].unique()
departamento_options = Demo['Departamento'].unique()

#Agregar una opción extra "Todos"
all_option = {"label": "Todos", "value": "Todos"}

options_with_all1 = [{"label": loc, "value": loc} for loc in locacion_options]
options_with_all1.insert(0, all_option)

options_with_all2 = [{"label": loc, "value": loc} for loc in funcion_options]
options_with_all2.insert(0, all_option)

options_with_all3 = [{"label": loc, "value": loc} for loc in departamento_options]
options_with_all3.insert(0, all_option)

## __Creation of the integrated dashboard__

In this case, the page to be displayed will feature a central section, and each dashboard will have its own predefined page as well as specific controls tailored to the type of chart being used.

In [ ]:
#Página numero uno
#Creacion de un tablero para la visualización inicial (solo texto)
def page1():
   return html.Div([
       html.Div(children=[
            html.Div(children=html.B(children=html.H4("Descripción del Reto")),
                     className="alert alert-dismissible alert-success",
                     style={"background-color": "#891B18", "color": "white"}),
            html.P("El reto de Liverpool se trata de poder predecir con precisión la futura"\
                   " renuncia de sus actuales empleados debido a varios factores históricos y demográficos,"\
                   " por medio de un análisis matemático y estadístico de los atributos de personal"\
                   " que ya renunció - dichos atributos engloban características tales como la función"\
                   " de un exempleado, su edad al momento de ingresar a la compañía, su año de nacimiento, entre otros.")],
            #Clase de control
            className="accordion-item"),

       html.Div(children=[
            html.Div(children=html.B(children=html.H4("Uso de Visualizaciones")),
                     className="alert alert-dismissible alert-success",
                     style={"background-color": "#891B18", "color": "white"}),
            html.P("En primera instancia, se hace uso de visualizaciones/gráficos sobre los datos de Liverpool"\
                   " para detectar elementos anómalos en la información a través de la creación de varias "\
                   " hipótesis sustentadas por una narrativa de datos y elementos básicos de estadística."\
                   " A través de estas visualizaciones podemos remover outliers, entender mejor nuestro"\
                   " conjunto de datos y eventualmente, crear un modelo de predicción que pueda clasificar futuras renuncias.")],
            #Clase de control
            className="accordion-item"),

       html.Div(children=[
            html.Div(children=html.B(children=html.H4("Machine Learning")),
                     className="alert alert-dismissible alert-success",
                     style={"background-color": "#891B18", "color": "white"}),
            html.P("Después de un análisis más avanzado, se utilizan técnicas de descubrimiento de patrones más"\
                   " sofisticadas basadas en el aprendizaje supervisado - en particular, dos clasificadores: Random Forest y"\
                   " Regresión Logística Multinomial (correspondientes a Primer Pronóstico y Segundo Pronóstico, respectivamente,"\
                   " en la sección 'Predicción Modelo').")],
            #Clase de control
            className="accordion-item"),

       html.Div(children=[
            html.Div(children=html.B(children=html.H4("Eficacia de los Modelos")),
                     className="alert alert-dismissible alert-success",
                     style={"background-color": "#891B18", "color": "white"}),
            html.P("Tras dividir nuestro set de datos en sets de entrenamiento y de prueba (80%-20%, respectivamente), obtuvimos los siguientes resultados."\
                   " En cuanto al primer modelo de predicción (random forest), hallamos que nuestro modelo predecía bien 8 de cada 10"\
                   " empleados (con un accuracy, precision y recall de 0.78) sobre el tiempo que iban a tardar en renunciar. Nuestro segundo modelo consiguió"\
                   " los mismos resultados. Ambos predicen que un empleado puede renunciar en menos de 1 año, pero más de cuatro meses;"\
                   " en más de 1 año, pero menos de 5 años; y en más de 5 años.")],
            #Clase de control
            className="accordion-item"),
       #Se cierra el contenedor de todo el tablero
   ])

In [ ]:
#Página numero dos
"""
Creación de un tablero para la predicción de los modelos con
un botón de carga del archivo y cinco celdas de texto que se actualizan automáticamente
"""
def page2():
  return html.Div([

   #Barra de titulo del tablero
   html.Div(children=[html.H1(children="Predicción Modelo",
                              style={"color": "black",
                                     "height": "57v"})]),
   html.Hr(style={"border-color": "black", "border-width": "3px"}),

   #Se añade en la primera fila con cinco tipos de controles, un botón de carga de archivo y un recuadro de estado
   dbc.Row([dbc.Col(children=[

      html.Div(children=[
            html.Div(children=html.B(children=html.H4("Botón de Carga")),
                     className="alert alert-dismissible alert-success",
                     style={"text-align": "center", "background-color": "#891B18", "color": "white"}),

            #Tipo de control
            dcc.Upload(id="uploadButton",
                       children= html.Button("Buscar archivo", className="btn btn-primary btn-sm"),
                       multiple=False,
                       style={
                              "width": "80%",  #Establece el ancho del botón
                              "height": "60px",  #Establece la altura del botón
                              "lineHeight": "60px",  #Establece la altura de línea
                              "borderWidth": "1px",  #Establece el ancho del borde
                              "textAlign": "center",  #Centra el texto
                              "margin": "auto"  #Centra el botón en la página
                              }
                       )
            ],
            style={"height": "27vh", #Establece la altura del marco
                   "width": "95%",  #Establece el ancho del marco
                   },
            #Clase del elemento
            className="accordion-item"),

      html.Div(children=[
            html.Div(children=html.B(children= html.H4("Nombre del Archivo")),
                     id="dinamicText",
                     style={"height": "25vh",# Establece la altura de la caja de texto
                            "width": "95%", #Establece el ancho de la caja de texto
                            "display": "flex",#Establece la posición de la caja de texto (modo dinamico)
                            "align-items":"center", #Centra la información de la caja de texto
                            "justify-content":"center",
                            "background-color": "#891B18",
                            "color": "white",
                            "border-radius": "3%"}),
            ],)

       ],
       style={"height": "57v",
              "margin-left" : "11px",
              "margin": "4px",}),

       #Se añade en la primera fila una segunda columna con áreas de texto
       dbc.Col(html.Div(children=[
           dbc.Row([
               #Area de texto 1
               dbc.Col(html.Div(children=html.B(children= html.H5(["Primer Pronóstico"])),
                     id="arearf1",
                     style={"border":"2px black solid",
                            "height": "25vh",
                            "width": "100%",
                            "display": "flex",
                            "align-items":"center",
                            "justify-content":"center",
                            "background-color": "#F4831B",
                            "box-shadow": "-5px -5px 10px 4px rgba(0, 0, 0, 0.25)"})),
               #Area de texto 2
               dbc.Col(html.Div(children=html.B(children= html.H5(["Probabilidades 1"])),
                     id="arearf2",
                     style={"border":"2px black solid",
                            "height": "25vh",
                            "width": "100%",
                            "display": "flex",
                            "align-items":"center",
                            "justify-content":"center",
                            "background-color": "#F4831B",
                            "box-shadow": "5px -5px 10px 4px rgba(0, 0, 0, 0.25)"}))],
                   style={"margin": "10px"}),

           dbc.Row([
               #Area de texto 3
               dbc.Col(html.Div(children=html.B(children= html.H5(["Segundo Pronóstico"])),
                     id="arealr1",
                     style={"border":"2px black solid",
                            "height": "25vh",
                            "width": "100%",
                            "display": "flex",
                            "align-items":"center",
                            "justify-content":"center",
                            "background-color": "#F4831B",
                            "box-shadow": "-5px 5px 10px 4px rgba(0, 0, 0, 0.25)"},
                     )),
               #Area de texto 4
               dbc.Col(html.Div(children=html.B(children= html.H5(["Probabilidades 2"])),
                     id="arealr2",
                     style={"border":"2px black solid",
                            "height": "25vh",
                            "width": "100%",
                            "display": "flex",
                            "align-items":"center",
                            "justify-content":"center",
                            "background-color": "#F4831B",
                            "box-shadow": "5px 5px 10px 4px rgba(0, 0, 0, 0.25)"}))],
                    style={"margin": "10px"}),

           dbc.Row([
              # Area de texto 5 (new box added after the fourth box)
               dbc.Col(html.Div(children=html.B(children=html.H5(["Pronóstico Definitivo",
                                                                  html.Br(),
                                                                  "(Probabilidad más Alta)"])),
                                id="areadef",
                                style={ "border":"3px black solid",
                                        "height": "25vh",
                                        "width": "60%",
                                        "display": "flex",
                                        "align-items": "center",
                                        "justify-content": "center",
                                        "background-color": "#FC993D",
                                        "margin": "10px auto",
                                        "border-radius": "15%",
                                        "box-shadow": "0px 7px 10px 3px rgba(0, 0, 0, 0.3)"},
                                ), className="mx-auto"),
            ]),

       #Tipo de elemento sobre el tablero
       ])),

   #Fin de la primera fila
   ]),

#Se cierra el contenedor de todo el tablero
])

In [ ]:
#Página numero tres
"""
Creación de un tablero para una visualización de cuatro gráficos con la
funcionalidad de plotly express y botones interactivos de dash considerando los features más relevantes
"""
def page3():
  return html.Div([

   #Barra de titulo del tablero
   html.Div(children=[html.H1(children="Gráficos Dataset Demográfico",
                              style={"color": "black",
                                     "height": "57v"})]),
   html.Hr(style={"border-color": "black", "border-width": "3px"}),

   #Primera fila con tres columnas de controladores
   dbc.Row([
            dbc.Col(html.Div(html.Div(children=[
            html.Div(children=html.B(children="Año de Salida"),
                     className="alert alert-dismissible alert-success",
                     style={"background-color": "#891B18", "color": "white"}),
            #Tipo de control
            dcc.Dropdown(id="dropDown1",
                         clearable=False,
                         value="Todos",
                         options=[{"label": c, "value": c} for c in ["Todos",2019,2020,2021,2022]])
            ],
            #Clase de control
            className="accordion-item"))),

            dbc.Col(html.Div(html.Div(children=[
            html.Div(children=html.B(children="Número de Hijos"),
                     className="alert alert-dismissible alert-success",
                     style={"background-color": "#891B18", "color": "white"}),
            #Tipo de control
            dcc.Slider(id="slider",
                       min=0,
                       max=4,
                       step=1,
                       value=4,  # Set the initial value
                       marks={i: f'{i}' for i in range(5)},),
            ],
            #Clase de control
            className="accordion-item"))),

            dbc.Col(html.Div(children=[
            html.Div(children=html.B(children="Mes de Salida"),
                     className="alert alert-dismissible alert-success",
                     style={"background-color": "#891B18", "color": "white"}),
            #Tipo de control
            dcc.Dropdown(id="dropDown2",
                         clearable=False,
                         value="Todos",
                         options=[{"label": c, "value": c} for c in ["Todos",1,2,3,4,5,6,7,8,9,10,11,12]])
            ],
            #Clase de control
            className="accordion-item")),
        ]
    ),

   #Segunda fila con tres columnas de controladores
   dbc.Row([
            dbc.Col(html.Div(html.Div(children=[
            html.Div(children=html.B(children="Función"),
                     className="alert alert-dismissible alert-success",
                     style={"background-color": "#891B18", "color": "white"}),
            #Tipo de control
            dcc.Dropdown(id="dropDown3",
                         clearable=False,
                         value="Todos",
                         options=options_with_all2),
            ],
            #Clase de control
            className="accordion-item"))),

            dbc.Col(html.Div(children=[
            html.Div(children=html.B(children="Departamento"),
                     className="alert alert-dismissible alert-success",
                     style={"background-color": "#891B18", "color": "white"}),
            #Tipo de control
            dcc.Dropdown(id="dropDown4",
                         clearable=False,
                         value="Todos",
                         options=options_with_all3)
            ],
            #Clase de control
            className="accordion-item")),
        ]
    ),
   html.Br(),

   #Se añaden dos gráficos en otra fila
   dbc.Row([
          dbc.Col(html.Div(dcc.Graph(id="scatterPlot1",
                                      config={"displayModeBar": False}),
                                      style={"border":"2px black solid",
                                             "box-shadow": "5px 5px 10px 2px rgba(0, 0, 0, 0.25)"},
                                      #Tipo de elemento sobre el tablero
                                      className="modal-content",),
                  width=6),
          dbc.Col(html.Div(dcc.Graph(id="linePlot",
                                      config={"displayModeBar": False}),
                                      style={"border":"2px black solid",
                                             "box-shadow": "-5px 5px 10px 2px rgba(0, 0, 0, 0.25)"},
                                      #Tipo de elemento sobre el tablero
                                      className="modal-content",),
                  width=6),
   ]),
   html.Br(),

   #Se añaden dos gráficos en otra fila
   dbc.Row([
          dbc.Col(html.Div(dcc.Graph(id="barPlot",
                                      config={"displayModeBar": False}),
                                      style={"border":"2px black solid",
                                             "box-shadow": "5px -5px 10px 2px rgba(0, 0, 0, 0.25)"},
                                      #Tipo de elemento sobre el tablero
                                      className="modal-content",),
                  width=6),
          dbc.Col(html.Div(dcc.Graph(id="scatterPlot2",
                                      config={"displayModeBar": False}),
                                      style={"border":"2px black solid",
                                             "box-shadow": "-5px -5px 10px 2px rgba(0, 0, 0, 0.25)"},
                                      #Tipo de elemento sobre el tablero
                                      className="modal-content",),
                  width=6),
   ]),

#Se cierra el contenedor de todo el tablero
])

In [ ]:
#Página numero cuatro
"""
Creación de un tablero para una visualización de dos gráficos auxiliares de línea
(uno general y otro con mucha información)con la funcionalidad de plotly express
"""
def page4():
  #Configurar y mostrar gráfico
  frecuencia_antiguedad = Demo.groupby('Antigüedad').size().reset_index(name='Frecuencia')
  figure1 = px.line(frecuencia_antiguedad,
              x="Antigüedad",
              y="Frecuencia",
              labels={"Antigüedad": "Antigüedad de los empleados (en años)", "Frecuencia": "Número de bajas"},
              title='Antigüedad vs Frecuencia de Bajas')
  figure1.update_traces(line=dict(color='white'))
  figure1.update_layout({
        'xaxis': {'gridcolor': '#BEBEBE', 'showgrid': True},
        'yaxis': {'gridcolor': '#BEBEBE', 'showgrid': True},
        'plot_bgcolor': '#FA5CBE'})

  #Se escogen las 5 funciones con más renuncias
  top5funciones=["Vendedor Cajero","Personal Operativo","Cajero","Auxiliar Cajero","Recepcion Mercancia"]
  #Se filtra el dataframe para quedarnos unicamente con las funciones antes mostradas
  dfFunciones = Demo[Demo["Desc Fun"].isin(top5funciones)]
  #Se filtra el dataframe para contar las ocurrencias de renuncias por género y función
  Frecuencia_funciones = dfFunciones.groupby(['SAño','Desc Fun']).size().reset_index(name='Número de Salidas')
  #Se crea multiples visualizacion de linea en plotly express
  figure2 = px.line(Frecuencia_funciones,
              template="seaborn",
              x="SAño",
              y="Número de Salidas",
              facet_col="Desc Fun",
              labels={"SAño": "Año de Renuncia", "Número de Salidas": "Frecuencia",
              "Desc Fun":"Función"},
              title="Análisis de Renuncias por Puesto en los Top 5 con Mayor Índice (2019-2023)")
  figure2.update_traces(line=dict(color='white'))
  figure2.update_xaxes(gridcolor='#BEBEBE', showgrid=True)
  figure2.update_yaxes(gridcolor='#BEBEBE', showgrid=True)
  figure2.update_layout(plot_bgcolor='#FA5CBE')

  return html.Div([
      #Barra de titulo del tablero
      html.Div(children=[html.H1(children="Gráficos Generales de Antigüedad y Función",
                                  style={"color": "black",
                                        "height": "57v"})]),
      html.Hr(style={"border-color": "black", "border-width": "3px"}),
      html.Br(),
      #Visualizacion de gráfico de línea
      html.Div(dcc.Graph(id="Plot1",
                        figure=figure1,
                        config={"displayModeBar": False}),
                        #Tipo de elemento sobre el tablero
                        className="modal-content"),
      html.Br(),
      #Visualizacion de gráfico de línea
      html.Div(dcc.Graph(id="Plot2",
                        figure=figure2,
                        config={"displayModeBar": False}),
                        #Tipo de elemento sobre el tablero
                        className="modal-content")
      ])

In [ ]:
#Creacion de la aplicacion principal de dash
app = JupyterDash(__name__,
                  prevent_initial_callbacks=True,
                  suppress_callback_exceptions=True,
                  external_stylesheets=[dbc.themes.BOOTSTRAP])

#Nombre del tablero dentro del navegador
app.title = "Dashboard Liverpool"

#Elementos de estilo para la barra lateral izquierda (CSS)
SIDEBAR_STYLE = {
    "position": "fixed",
    "top": 0,
    "left": 0,
    "bottom": 0,
    "width": "16rem",
    "padding": "2rem 1rem",
    "background-color": "#D30484",
    "box-shadow": "11px 0px 10px rgba(0, 0, 0, 0.3)",
}

#Elementos de estilo para la parte principal/central de la pagina
CONTENT_STYLE = {
    "margin-left": "18rem",
    "margin-right": "2rem",
    "padding": "2rem 1rem",
}

#Creacion de la barra lateral izquierda
sidebar = html.Div(
    [
        #Imagen del logo de la empresa
        html.Img(src="https://drive.google.com/uc?export=download&id=17nNRKkdnDI_crtmWkNDBbFQC1cw926NM",
                 style={"height": "auto", "width": "175px", "margin": "10px auto 10px 9%"}),
        html.H2("Liverpool", className="display-4", style={"color": "white"}),
        html.Hr(style={"color": "white"}),
        html.H5(
            "People Analytics", className="alert alert-dismissible alert-success", style={"text-align": "right", "background-color": "#000000", "color": "white"}
        ),
        #Ligamos los controles con las páginas a cargar desde el servidor
        dbc.Nav(
            [
                dbc.NavLink("Introducción", href="/", active="exact", style={"color": "white"}),
                dbc.NavLink("Predicción Modelo", href="/page-2", active="exact", style={"color": "white"}),
                dbc.NavLink("Gráficos Demográfico", href="/page-3", active="exact", style={"color": "white"}),
                dbc.NavLink("Gráficos Auxiliares", href="/page-4", active="exact", style={"color": "white"}),
            ],
            vertical=True,
            pills=True,
        ),
    ],
    style=SIDEBAR_STYLE
)

#Creacion del tablero principal que se actualizara con cada opcion escogida
content = html.Div(id="page-content", style=CONTENT_STYLE)
app.layout = html.Div([dcc.Location(id="url", refresh=False), sidebar, content])

#Funcion activadora para unir cada subtablero con la pagina principal
@app.callback(Output("page-content", "children"), [Input("url", "pathname")])
def render_page_content(pathname):
    if pathname == "/":
        return page1()
    elif pathname == "/page-2":
        return page2()
    elif pathname == "/page-3":
        return page3()
    elif pathname == "/page-4":
        return page4()

    #Si el usuario usa una página diferente, se enviara un mensaje 404
    return html.Div(
        [
            html.H1("404: Página no encontrada", className="text-danger"),
            html.Hr(),
            html.P(f"La URL {pathname} no es reconocida..."),
        ],
        className="p-3 bg-light rounded-3",
    )


#Se crea el decorador y la función que manejaran la interactividad del botón de carga
#Definimos entradas y salidas
@app.callback(Output("dinamicText", "children"),
              Output("arearf1", "children"),
              Output("arearf2", "children"),
              Output("arealr1", "children"),
              Output("arealr2", "children"),
              Output("areadef", "children"),
              Input("uploadButton", "contents"),
              State("uploadButton", "filename"),
              State("uploadButton", "last_modified"))
#Definimos la función que implementara el proceso interactivo
def updatePage2(content, name, date):
   #Obtenemos el contexto de ejecución de dash
   ctx = dash.callback_context
   #Si la pagina se crea por primera vez hacemos lo siguiente
   if not(ctx.triggered):
    #Nombre genérico de la caja dinámica y las cajas de predicción
    dinamictext=html.H4("Nombre del Archivo")
    text1=html.H5(["Primer Pronóstico",html.Br()])
    text2=html.H5(["Probabilidades 1",html.Br()])
    text3=html.H5(["Segundo Pronóstico",html.Br()])
    text4=html.H5(["Probabilidades 2",html.Br()])
    text5=html.H5(["Pronóstico Definitivo",html.Br(),"(Probabilidad más Alta)"])
   #La pagina ya esta creada y se uso el boton
   else:
    #Se selecciono un archivo del botón de carga de archivos
    if content is not None:
      try:
        dinamictext= html.H4(["Nombre del Archivo:",html.Br(),name])
        contentType, contentString = content.split(',')
        decoded = base64.b64decode(contentString)

        #Archivo clasico donde se guradan dataframes de pandas
        if "csv" in name:
          DatasetLiverpool = pd.read_csv(io.StringIO(decoded.decode('utf-8')))
          text1, text2, text3, text4, text5 = predict_and_display_results(DatasetLiverpool, transformations, standard_scaler, LogisticRegression_clf, RandomForest_clf, class_mapping, class_labels)

        #Archivo de excel
        elif "xls" in name:
          DatasetLiverpool = pd.read_excel(io.BytesIO(decoded))
          text1, text2, text3, text4, text5 = predict_and_display_results(DatasetLiverpool, transformations, standard_scaler, LogisticRegression_clf, RandomForest_clf, class_mapping, class_labels)

        else:
           #Nombre genérico de la caja dinámica
           dinamictext= html.H4(["Nombre del Archivo:",html.Br(),"Error al cargar archivo,",html.Br(),"formato incorrecto.",html.Br(),"Porfavor inserte archivo",html.Br(),"en .csv o .xlsx"])
           text1=html.H5(["Primer Pronóstico",html.Br()])
           text2=html.H5(["Probabilidades 1",html.Br()])
           text3=html.H5(["Segundo Pronóstico",html.Br()])
           text4=html.H5(["Probabilidades 2",html.Br()])
           text5=html.H5(["Pronóstico Definitivo",html.Br(),"(Probabilidad más Alta)"])
      #En caso de que se seleccione un archivo que no sea soportado
      except Exception as e:
        #Nombre genérico de la caja dinámica
        dinamictext= html.H4(["Nombre del Archivo:",html.Br(),"Error al cargar archivo,",html.Br(),"formato incorrecto.",html.Br(),"Porfavor inserte archivo",html.Br(),"en .csv o .xlsx"])
        text1=html.H5(["Primer Pronóstico",html.Br()])
        text2=html.H5(["Probabilidades 1",html.Br()])
        text3=html.H5(["Segundo Pronóstico",html.Br()])
        text4=html.H5(["Probabilidades 2",html.Br()])
        text5=html.H5(["Pronóstico Definitivo",html.Br(),"(Probabilidad más Alta)"])

   #Se regresa la información de la caja de texto
   return dinamictext,text1,text2,text3,text4,text5


#Funcion activadora para el gráfico de dispersión
@app.callback(
    #Graficos que se activaran
    Output("scatterPlot1", "figure"),
    #Entrada de datos para los graficos
    [Input("dropDown1", "value"),
     Input("slider", "value"),]
)
def updateFigure1(year, kid):
   if (year=="Todos"):
      filtered_data = Demo.query("`No Hijos` <= "+str(kid))
   else:
      filtered_data = Demo[Demo['SAño'] == int(year)]
   # Calcular la frecuencia de salida por año en el DataFrame filtrado
   frecuencia_ano_salida = filtered_data.groupby('SAño').size().reset_index(name='Frecuencia')
   # Crear el grafico en plotly express
   fig1 = px.pie(frecuencia_ano_salida,
            values='Frecuencia',
            names='SAño',
            hole=0.3,
            color_discrete_sequence=px.colors.sequential.Plasma,
            labels={"SAño": "Año de Salida",
            "Frecuencia": "Cantidad de Renuncias"},
            title="Años de Egreso de los Empleados y Cantidad de Renuncias")
   return fig1

#Funcion activadora para el gráfico de línea
@app.callback(
    #Graficos que se activaran
    Output("linePlot", "figure"),
    #Entrada de datos para los graficos
    [Input("dropDown2", "value"),
     Input("slider", "value"),]
)
def updateFigure2(month, kid):
   if (month=="Todos"):
      filtered_data2 = Demo.query("`No Hijos` <= "+str(kid))
   else:
      filtered_data2 = Demo[Demo['SMes'] == int(month)]
   # Calcular la frecuencia de salida por mes en el DataFrame filtrado
   frecuencia_mes = filtered_data2.groupby('SMes').size().reset_index(name='Frecuencia')
   # Se crea el gráfico de línea en plotly express
   fig2 = px.line(frecuencia_mes,
                x="SMes",
                y="Frecuencia",
                labels={"SMes": "Mes", "Frecuencia": "Número de bajas"},
                title='Mes de Egreso vs Frecuencia de Bajas')
   fig2.update_traces(line=dict(color='#FA5CBE'))
   fig2.update_xaxes(dtick=1)  # Evitar valores intermedios en el eje x
   return fig2

#Funcion activadora para el gráfico de barras
@app.callback(
    #Graficos que se activaran
    Output("barPlot", "figure"),
    #Entrada de datos para los graficos
    [Input("dropDown3", "value"),
     Input("slider", "value"),]
)
def updateFigure3(function, kid):
   if (function=="Todos"):
      filtered_data3 = Demo.query("`No Hijos` <= "+str(kid))
   else:
      filtered_data3 = Demo[Demo['Desc Fun'] == str(function)]
   #Calcular la frecuencia de salida por función en el DataFrame filtrado
   frecuencia_funcion = filtered_data3.groupby('Desc Fun').size().reset_index(name='Frecuencia')
   frecuencia_funcion = frecuencia_funcion.sort_values(by='Frecuencia', ascending=False).head(20)
   fig3 = px.bar(frecuencia_funcion,
            x="Desc Fun",
            y="Frecuencia",
            color="Frecuencia",
            color_continuous_scale='Viridis',
            labels={"Desc Fun": "Nombre de la función",
            "Frecuencia": "Frecuencia"},
            title="Renuncias por Función")
   fig3.update_xaxes(tickangle=45)
   #Se crea el gráfico en plotly express
   return fig3

#Funcion activadora para el segundo gráfico de dispersión
@app.callback(
    #Graficos que se activaran
    Output("scatterPlot2", "figure"),
    #Entrada de datos para los graficos
    [Input("dropDown4", "value"),
     Input("slider", "value"),]
)
def updateFigure4(department, kid):
   if (department=="Todos"):
      filtered_data4 = Demo.query("`No Hijos` <= "+str(kid))
   else:
      filtered_data4 = Demo[Demo['Departamento'] == str(department)]

   df = filtered_data4.groupby('Departamento').size().reset_index(name='Número de Bajas')
   fig4 = px.scatter(df,
              y='Departamento',
              x='Número de Bajas',
              size='Número de Bajas',
              color='Número de Bajas',
              title='Renuncias por Departamentos' ,
              labels={'Departamento': 'Departamento', 'Número de Bajas': 'Renuncias'},
              color_continuous_scale='Magma')

   fig4.update_layout(yaxis_tickangle=-45)
   #Se crea el grafico en plotly express
   return fig4

#Se inicializa la aplicacion de dash
app.run_server(host="127.0.0.1", port=8027, debug=True)

<IPython.core.display.Javascript object>

Dash app running on:


<IPython.core.display.Javascript object>